# Macquarie Coal Complex Transformation Precinct — Spatial ETL

Master notebook for the precinct pipeline.
- Stage 1: environmentsetup + repo install
- Stage 2: storage root / CRS / buffers
- Stage 3: ingest Lake Mac / NSW SEED / ABS / TfNSW
- Stage 4: constraints overlay → net developable zones
- Stage 5: Wherobots introspection / validation

In [ ]:
%pip install --quiet geopandas pyproj requests pandas
from sedona.spark import *
spark = SedonaContext.create(SedonaContext.builder().getOrCreate())
spark.sparkContext.setLogLevel("INFO")
print("Sedona ready.")

In [ ]:
import json, os
from hunter_spatial_crafter.src.Ingestion.macquarie_spatial_ingest import _cfg, load_precinct_boundary

cfg = _cfg()
S3_BUCKET = cfg.get("storage_root", "wherobots://fgsdb/macquarie")
TARGET_CRS = cfg.get("target_crs", "EPSG:7856")
SOURCE_CRS = cfg.get("source_crs", "EPSG:4326")

print(f"Storage root : {S3_BUCKET}")
print(f"Source CRS   : {SOURCE_CRS}")
print(f"Target CRS   : {TARGET_CRS}")
print(f"Sub-precincts: {cfg['precinct']['sub_precincts']}")

In [ ]:
load_precinct_boundary(spark, S3_BUCKET)
spark.table("org_catalog.fgsdb.macquarie_precinct_boundary").show(truncate=False)

In [ ]:
from hunter_spatial_crafter.src.Ingestion.macquarie_spatial_ingest import (
    load_water_infrastructure,
    load_biodiversity_constraints,
    load_energy_infrastructure,
    load_pipeline_corridors,
    load_transport_networks,
    load_abs_meshblocks,
    build_net_developable_zones,
    run_verification,
)

load_water_infrastructure(spark, S3_BUCKET, cfg)
load_biodiversity_constraints(spark, S3_BUCKET, cfg)
load_energy_infrastructure(spark, S3_BUCKET, cfg)
load_pipeline_corridors(spark, S3_BUCKET, cfg)
load_transport_networks(spark, S3_BUCKET, cfg)
load_abs_meshblocks(spark, S3_BUCKET, cfg)
print("Open-data layers ingested.")

In [ ]:
build_net_developable_zones(spark, S3_BUCKET, cfg)

zones_df = spark.table("org_catalog.fgsdb.macquarie_net_developable_zones")
zones_df.selectExpr(
    precinct_key,
    ST_Area(net_developable_geom) / 1e4 AS developable_ha
).show(truncate=False)
run_verification(spark)

## Data Flow Summary

- **Lake Mac Open Data** → `macquarie_active_transport`
- **NSW SEED** → `macquarie_water_hydrography`, `macquarie_biodiversity_constraints`
- **NSW Spatial Services** → `macquarie_energy_infrastructure`, `macquarie_pipeline_corridors`, `macquarie_rail_network`
- **ABS Digital Atlas** → `macquarie_abs_meshblocks`
- **Precinct + Constraints** → `macquarie_net_developable_zones`